# Limpeza dos dados

Título da pesquisa: Predição de Acidentes em Rodovias Federais de Santa Catarina por Meio de Séries Temporais e Aprendizado de Máquina.

Apesar da PRF disponibilizar dados desde 2007, para nossa análise consideraremos os dados de 2022 a 2026, visto que ao longo dos anos as rodovias sofreram muitas mudanças e também que em 2020 e 2021 a pandemia impactou na quantidade de veículos nas rodovias, alterando padrões de acidentes.

## Importações e carregamento dos dados

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import unicodedata
import glob

In [2]:
# Carrega tabelas, filtra por somente acidentes em que uf == "SC" e concatena todas em uma para manipulação facilitada

caminho_ficheiros = '../data/raw/datatran*.csv'
lista_ficheiros = sorted(glob.glob(caminho_ficheiros))

dfs = []

for ficheiro in lista_ficheiros:
    # Lê o CSV ajustando separador e codificação típicos da PRF
    df_temp = pd.read_csv(
        ficheiro, sep=';', encoding='latin1', low_memory=False
    )

    # Filtra por UF == 'SC' logo na leitura para economizar memória
    if 'uf' in df_temp.columns:
        df_temp = df_temp[
            df_temp['uf'].astype(str).str.strip().str.upper() == 'SC'
        ]

    dfs.append(df_temp)

# Concatena todas as tabelas numa só
acidentes_SC = pd.concat(dfs, ignore_index=True)

## Funções

In [7]:
# FUNÇÃO PARA TRANSFORMAR STRING PARA LOWER CASE E REMOÇÃO DE ACENTOS

def lower_case_pd(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """
    Função para transformar os valores de uma ou mais colunas de um DataFrame em letras minúsculas
    e remover acentos.

    Parâmetros:
    df (pd.DataFrame): O DataFrame que contém as colunas a serem transformadas.
    colunas (list): Uma lista de nomes de colunas que devem ser convertidas para letras minúsculas.
    
    Retorna:
    pd.DataFrame: O DataFrame com as colunas especificadas convertidas para letras minúsculas
    e sem acentos.
    """

    def remover_acentos_e_cedilha(texto) -> str:
        # Garante a conversão do valor para string caso seja float/NaN
        texto_str = str(texto) if pd.notna(texto) else ""

        # Substitui 'ç' e 'Ç' por 'c' e 'C'
        texto_str = texto_str.replace("ç", "c").replace("Ç", "C")

        # Normaliza o texto para decompor acentos (NFD) e remove as marcas diacríticas
        texto_normalizado = unicodedata.normalize("NFD", texto_str)
        return "".join(
            c
            for c in texto_normalizado
            if unicodedata.category(c) != "Mn"
        )

    for coluna in colunas:
        df[coluna] = (
            df[coluna]
            .astype(str)
            .str.lower()
            .str.strip()
            .apply(remover_acentos_e_cedilha)
        )

    return df

## Limpeza dos dados

In [4]:
# TRANSFORMACAO DE DATA INVERSA PARA DATETIME E PADRONIZAÇÃO PARA AAAA-MM-DD
# 1. Converte a coluna com formatos mistos para o tipo datetime
acidentes_SC['data_inversa'] = pd.to_datetime(acidentes_SC['data_inversa'], format='mixed', dayfirst=True, errors='coerce')

In [5]:
# TRANSFORMAÇÃO DE ID PARA TIPO INT

acidentes_SC['id'] = acidentes_SC['id'].astype('Int64')

In [8]:
# NORMALIZAÇÃO DE STRINGS PARA LOWER CASE

acidentes_SC = lower_case_pd(acidentes_SC, ['municipio', 'dia_semana', 
                                            'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 
                                            'fase_dia', 'condicao_metereologica', 'regional', 'delegacia'])


In [9]:
# REMOÇÃO DE COLUNAS DESNECESSÁRIAS PARA ANÁLISE

acidentes_SC = acidentes_SC.drop(columns= ['horario', 'km', 'tipo_pista', 'tracado_via', 
                                           'uso_solo', 'pessoas', 'ilesos', 'ignorados', 
                                           'latitude', 'longitude', 'uop', 'uf', 'sentido_via'])

## Carregamento para parquet

In [10]:
acidentes_SC.to_parquet(Path('../data/interim/acidentes_SC.parquet'), index=False)

print(f"Arquivo Parquet salvo com sucesso em data/interim")

Arquivo Parquet salvo com sucesso em data/interim
